# KanjiComparator — Siamese CNN Training
Run cells top to bottom. Make sure **Runtime → Change runtime type → T4 GPU** is selected before starting.

## 1. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')

## 2. Clone repository
Push your code to GitHub first, then replace the URL below.

In [ ]:
!git clone https://github.com/YOUR_USERNAME/KanjiComparator.git
%cd KanjiComparator

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Mount Google Drive and link dataset
Your dataset is read directly from Drive — no copying, no extra disk usage.
The symlink makes it appear at `kanji-dataset/` so the code needs no changes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_DATASET_PATH = '/content/drive/MyDrive/kanji-dataset'

# Symlink Drive folders to the path the code expects
os.makedirs('kanji-dataset', exist_ok=True)

for subfolder in ['referenceKanji', 'handwritingKanji']:
    src = os.path.join(DRIVE_DATASET_PATH, subfolder)
    dst = os.path.join('kanji-dataset', subfolder)
    if not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'Linked: {dst} → {src}')
    else:
        print(f'Already linked: {dst}')

## 5. Verify dataset structure
Expected layout:
```
kanji-dataset/
  referenceKanji/    ← PNG files named by unicode (e.g. 0f9a8.png)
  handwritingKanji/  ← numbered PNGs + meta.csv
```

In [ ]:
import os
ref = os.listdir('kanji-dataset/referenceKanji')
hw  = os.listdir('kanji-dataset/handwritingKanji')
print(f'Reference kanji:   {len([f for f in ref if f.endswith(".png")])} PNGs')
print(f'Handwriting kanji: {len([f for f in hw  if f.endswith(".png")])} PNGs')
print(f'meta.csv present:  {"meta.csv" in hw}')

## 6. Train

In [ ]:
!python train.py

## 7. View plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

plot_files = sorted([f for f in os.listdir('plots') if f.endswith('.png')])
fig, axes = plt.subplots(1, len(plot_files), figsize=(6 * len(plot_files), 5))
if len(plot_files) == 1:
    axes = [axes]
for ax, fname in zip(axes, plot_files):
    ax.imshow(mpimg.imread(f'plots/{fname}'))
    ax.set_title(fname.replace('_', ' ').replace('.png', ''))
    ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Save model to Drive

In [ ]:
import shutil, os

DRIVE_SAVE_PATH = '/content/drive/MyDrive/kanji-dataset/kanji_model.pt'
shutil.copy('mlmodel/kanji_model.pt', DRIVE_SAVE_PATH)
print(f'Model saved to Drive: {DRIVE_SAVE_PATH}')